# Load Raw Data Into Delta Tables

In [0]:
%python
# Import configuration from bronze_config.py
import sys
sys.path.append('/Workspace/Users/samwwalks@gmail.com/databricks_bootcamp_dwb/bike_lakehouse/bronze')

from bronze_config import (
    BASE_VOLUME_PATH,
    TARGET_SCHEMA,
    SOURCE_CONFIGS,
    get_table_name,
    get_full_table_name,
    get_all_csv_files,
    get_ingestion_config
)

print("Configuration imported successfully!")
print(f"Base Volume Path: {BASE_VOLUME_PATH}")
print(f"Target Schema: {TARGET_SCHEMA}")
print(f"Number of source configurations: {len(SOURCE_CONFIGS)}")

In [0]:
%python
# Get the ingestion configuration using the imported function
ingestion_config = get_ingestion_config(dbutils)

print(f"Found {len(ingestion_config)} CSV files to process:\n")
print("=" * 120)
print(f"{'Source Folder':<20} {'File Name':<35} {'Target Table':<65}")
print("=" * 120)

for item in ingestion_config:
    print(f"{item['source_folder']:<20} {item['file_name']:<35} {item['full_table_name']:<65}")

print("=" * 120)
print(f"\nTotal files to process: {len(ingestion_config)}")

In [0]:
%python
# Process all CSV files and write to Delta tables using imported configuration
from pyspark.sql import SparkSession

# Get the ingestion configuration
ingestion_config = get_ingestion_config(dbutils)

print(f"Starting ingestion of {len(ingestion_config)} files...\n")
print("=" * 100)

success_count = 0
failure_count = 0

# Process each file using the configuration
for item in ingestion_config:
    try:
        print(f"\nProcessing: {item['file_name']}")
        print(f"  Source: {item['source_file_path']}")
        print(f"  Target: {item['full_table_name']}")
        
        # Read CSV file
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(item['source_file_path'])
        
        # Write to Delta table using the full table name from config
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(item['full_table_name'])
        
        record_count = df.count()
        print(f"  ✓ Successfully loaded {record_count} records")
        success_count += 1
        
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        failure_count += 1

print("\n" + "=" * 100)
print(f"\nIngestion Summary:")
print(f"  ✓ Successful: {success_count}")
print(f"  ✗ Failed: {failure_count}")
print(f"  Total: {len(ingestion_config)}")
print("\nIngestion complete!")

In [0]:
-- Verify all tables were created in the bronze schema
SHOW TABLES IN databricks_bootcamp_dwb.bronze;